# 01｜认识行情数据
只读认识本项目的 SQLite 行情。选择项目 py311 内核，从上到下运行。本练习固定读取 2020-01-01 至 2020-02-28 按代码排序的五只股票。

In [ ]:
from pathlib import Path
import sys
ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "qlib_quant").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请在 VS Code 打开整个 D:/qlib_quant 文件夹。")
sys.path.insert(0, str(ROOT / "src"))
from qlib_quant.config import effective_config
from qlib_quant.data.db import list_codes, iter_quotes
cfg = effective_config()
DB = Path(cfg["data"]["database"])
print("解释器:", sys.executable, "\n项目:", ROOT, "\n数据库:", DB, "\n存在:", DB.exists())
assert DB.exists(), "请先在工作台首页检查数据库路径。"

## 字段对照
SQLite 表 `daily_quote` 的行情经项目读取函数标准化。数据库由只读连接打开，本练习不会写入。

In [ ]:
field_map = {
    "stock_code": "code", "trade_date": "date",
    "open_price": "open", "high_price": "high", "low_price": "low",
    "close_price": "close", "volume": "volume", "turnover": "amount",
    "trade_status": "trade_status",
}
field_map

In [ ]:
START, END = "2020-01-01", "2020-02-28"
codes = list_codes(DB, START, END, max_stocks=5)
import pandas as pd
df = pd.DataFrame(list(iter_quotes(DB, START, END, codes=codes)))
print("固定股票:", codes, "\n行情行数:", len(df), "\n字段:", list(df.columns))
display(df.head(10))

## 检查与练习
一行表示一只股票在一个交易日的一条行情。价格为后复权价，成交量和成交额按数据库口径保存。回测预热期由运行服务另行读取，本节查看的是绩效区间。

In [ ]:
assert df.code.dtype == object, "股票代码要作为字符串"
assert not df.duplicated(["code", "date"]).any(), "发现同股同日重复行情"
ordered = df.sort_values(["code", "date"])
assert ordered.groupby("code")["date"].apply(lambda s: s.is_monotonic_increasing).all()
print("通过：代码为字符串、没有重复、日期有序。")
one_code = codes[0]
display(ordered.loc[ordered.code == one_code,
                    ["code","date","open","high","low","close","volume","amount"]].tail(10))

### 完成检查
- [ ] 我能说出数据库路径和行情表名。
- [ ] 我能解释一行行情的含义及字段映射。
- [ ] 我理解预热数据只用于因子计算，不计入回测收益。
- [ ] 我能定位解释器、项目目录和数据库。

若导入失败，请确认 VS Code 打开的是项目根目录并选择 py311 内核。